In [0]:
from pyspark.sql import functions as F
from pyspark.sql import SparkSession
from pyspark.sql.types import StringType, DateType
from pyspark.sql.functions import *

# Create Spark session
spark = SparkSession.builder.appName("Validation").getOrCreate()

# Sample DataFrame (Replace with actual data loading method)
data = [
    ("1234", "2024-10-01", "2024-10-30", "pass_1"),
    ("123", "2023-10-05", "2024-10-04", "pass_2"),
    ("ADE15678", "2022-12-05", "2024-12-23", None),
    ("5678", "2017-10-02", "2024-09-21", "pass_3"),
    ("-1", "2018-03-08", "2019-12-10", "pass_4"),
    ("Emp02", "2019-11-24",	"2018-02-04", "pass_2"),
    ("0", "2021-12-01",	"2023-10-18", None),
    (None, "2016-10-14", "2015-01-23", "pass_5"),
    ("1749", "2012-07-20", "2025-10-01", "pass_3"),
    (None, "2020-02-09", "2024-11-13", "pass_2"),
    ("Emp6", "2000-10-30", "2025-04-27", "pass_10"),
    ("Emp8", "2022-11-25", "2019-02-14", "pass_7"),
    ("Emp@0543", "2002-03-11", "2001-01-01", None),
    ("%#4!", "", "", "pass_9")
]

columns = ["emp_id", "date_1", "date_2", "pass_id"]

# Create DataFrame
df = spark.createDataFrame(data, columns)

# Cast the date columns to DateType
df = df.withColumn("date_1", F.col("date_1").cast(DateType())) \
       .withColumn("date_2", F.col("date_2").cast(DateType()))

# Apply conditions
df_filtered = df.withColumn("emp_id_length", F.length(F.col("emp_id"))) \
    .withColumn("emp_id_valid", F.col("emp_id_length") == 4) \
    .withColumn("date_valid", F.col("date_1") <= F.col("date_2")) \
    .withColumn("emp_id_not_4", F.col("emp_id") != "4") \
    .withColumn("pass_id_not_null", F.col("pass_id").isNotNull()) \
    .withColumn("emp_id_status", 
                F.when(F.col("emp_id_valid") & 
                       F.col("date_valid") & 
                       F.col("emp_id_not_4") & 
                       F.col("pass_id_not_null"), 
                       F.lit("Valid"))  
                .otherwise(F.lit("Invalid"))  
                )
df_filtered.display()

emp_id,date_1,date_2,pass_id,emp_id_length,emp_id_valid,date_valid,emp_id_not_4,pass_id_not_null,emp_id_status
1234,2024-10-01,2024-10-30,pass_1,4,true,true,true,true,Valid
123,2023-10-05,2024-10-04,pass_2,3,false,true,true,true,Invalid
ADE15678,2022-12-05,2024-12-23,null,8,false,true,true,false,Invalid
5678,2017-10-02,2024-09-21,pass_3,4,true,true,true,true,Valid
-1,2018-03-08,2019-12-10,pass_4,2,false,true,true,true,Invalid
Emp02,2019-11-24,2018-02-04,pass_2,5,false,false,true,true,Invalid
0,2021-12-01,2023-10-18,null,1,false,true,true,false,Invalid
null,2016-10-14,2015-01-23,pass_5,null,null,false,null,true,Invalid
1749,2012-07-20,2025-10-01,pass_3,4,true,true,true,true,Valid
null,2020-02-09,2024-11-13,pass_2,null,null,true,null,true,Invalid
